### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
from collections import Counter
import xml.etree.ElementTree as ET
import folium
import folium.plugins
import branca
import branca.colormap as cm
import param
import panel as pn

### Data Preparation 

In [2]:
df = pd.read_parquet("data/map_matched_routes.parquet")
df = df[df.matched == True] # only matched routes
df = df.reset_index()

In [3]:
d=[]
for i in range(len(df)):
    d.append(df.unique_id[i][-10:].split('-')[0]+df.unique_id[i][-10:].split('-')[1]+df.unique_id[i][-10:].split('-')[2])
df['dates'] = d
df['dates'] = pd.to_datetime(df['dates'], format='%Y%m%d')

In [4]:
df['month'] = df['dates'].dt.month
df['year'] = df['dates'].dt.year
df = df.reset_index()
df = df.drop(['level_0', 'index'], axis=1)
df

,unique_id,route,matched,dates,month,year
0,00000bf0-0000-1000-a6a0-5b7d6a617b60_2020-11-30,"[[46.069691, 11.121241], [46.069716, 11.121509...",True,2020-11-30,11,2020
1,00001d90-0000-1000-8b30-5b7d6a617b60_2020-11-30,"[[46.07016, 11.120398], [46.070181, 11.12065],...",True,2020-11-30,11,2020
2,0000210d-0000-1000-8d7d-5b7d6a617b60_2020-11-30,"[[46.067214, 11.126787], [46.067211, 11.126772...",True,2020-11-30,11,2020
3,00000735-0000-1000-9635-5b7d6a617b60_2020-11-30,"[[46.069691, 11.12124], [46.069699, 11.121327]]",True,2020-11-30,11,2020
4,00000e00-0000-1000-9ab0-5b7d6a617b60_2020-11-30,"[[46.069695, 11.121283], [46.069716, 11.121509...",True,2020-11-30,11,2020
...,...,...,...,...,...,...
70416,00000c1f-0000-1000-a6bf-5b7d6a617b60_2022-03-04,"[[46.063361, 11.12368], [46.063356, 11.123623]...",True,2022-03-04,3,2022
70417,830baf6d-455a-4f22-bdd4-9be7b521622d_2022-03-04,"[[46.075298, 11.123749], [46.075298, 11.123749]]",True,2022-03-04,3,2022
70418,0daf00f7-4afb-43c6-8c7f-5e674945192d_2022-03-04,"[[46.075385, 11.123814], [46.075445, 11.123859...",True,2022-03-04,3,2022
70419,00000d44-0000-1000-9a34-5b7d6a617b60_2022-03-04,"[[46.070977, 11.124824], [46.070926, 11.124821...",True,2022-03-04,3,2022


In [5]:
df.groupby(['month','year']).size().reset_index().rename(columns={0:'count'}).sort_values(by=['year', 'month']).reset_index().drop(['index'], axis=1)

,month,year,count
0,11,2020,20
1,12,2020,2310
2,1,2021,1149
3,2,2021,2962
4,3,2021,4862
5,4,2021,4886
6,5,2021,5978
7,6,2021,7112
8,7,2021,6921
9,8,2021,5806


In [15]:
# Valhalla request
edge_ids=[]
ids = df.unique_id.to_list()
for id in range(len(ids)):
    df_temp = df[df.unique_id==ids[id]]
    df_points = pd.DataFrame({'lon':[el[1] for i in df_temp.route for el in i], 'lat':[el[0] for i in df_temp.route for el in i]})

    # request
    meili_coordinates = df_points.to_json(orient='records')
    meili_head = '{"shape":'
    meili_tail = ""","search_radius": 300, "shape_match":"edge_walk", "costing":"bicycle", "format":"osrm"}""" # use 'shape_match' : 'edge_walk'
    meili_request_body = meili_head + meili_coordinates + meili_tail
    url = "http://localhost:8002/trace_attributes" # use /trace_attributes 
    headers = {'Content-type': 'application/json'}
    data = str(meili_request_body)

    r = requests.post(url, data=data, headers=headers)
    
    response_text = json.loads(r.text)

    if r.status_code == 200:
        l=[]
        for i in range(len(response_text['edges'])):
            l.append(response_text['edges'][i]['way_id']) #, response_text['osm_changeset'])

        edge_ids.append(list(set(l)))
    else:
        edge_ids.append([])  

In [16]:
df['edge_ids'] = edge_ids
df = df[df['edge_ids'].map(lambda d: len(d)) > 0] # remove rows where edge_ids is empty

In [37]:
df.to_parquet('map_matched_edges_ids.parquet')

In [20]:
c = []
for el in edge_ids:
    c.extend(el)
c = Counter(c)

els = 'id: '
for el in c.keys():
    els = els + str(el) +', '
els = els[:-2] # string of way ids to pass to request coordinates

In [23]:
# Overpass request
url = 'http://localhost:12345/api/interpreter'
data = 'data=way('+str(els)+');out geom;'
r = requests.post(url, data=data)
root = ET.fromstring(r.text)

nodes_coords = []
waysids = []
for child in root:
    if child.tag == 'way':
        waysids.append(int(child.attrib['id']))
        nodes=[]
        for node in child:
            if node.tag == 'nd':
                # print(node.attrib['ref']) # node id
                nodes.append([float(node.attrib['lat']), float(node.attrib['lon'])])
        nodes_coords.append(nodes)

In [28]:
cnt=[]
for id in waysids:
    cnt.append(c[id])

In [29]:
ways = pd.DataFrame({'way_id': waysids, 'route': nodes_coords, 'freq': cnt})
ways  

,way_id,route,freq
0,22986084,"[[46.0666524, 11.1190333], [46.0666403, 11.118...",3521
1,22986312,"[[46.0674662, 11.1266008], [46.0675156, 11.126...",2309
2,22986560,"[[46.069021, 11.1225351], [46.0693306, 11.1224...",1583
3,23056771,"[[46.0969958, 11.1125195], [46.096235, 11.1127...",26
4,23058066,"[[46.0838565, 11.1146839], [46.0839814, 11.114...",287
...,...,...,...
3621,1059368470,"[[46.1051947, 11.1092483], [46.1053035, 11.109...",4
3622,1059368474,"[[46.0997827, 11.1128606], [46.099779, 11.1137...",1
3623,1059368478,"[[46.1056222, 11.1147083], [46.1058355, 11.114...",2
3624,1059368479,"[[46.1061001, 11.115034], [46.105945, 11.11482...",3


In [34]:
# save ways ids, count, and coordinates as json 
# way_id (keys) :  {coords: [list of lists], count_complessivo: int}
d={}
for i in range(len(waysids)):
    d[waysids[i]] = {}
    d[waysids[i]]['coords'] = nodes_coords[i]
    d[waysids[i]]['count_complessivo'] = cnt[i]

In [35]:
d

{22986084: {'coords': [[46.0666524, 11.1190333],
   [46.0666403, 11.1189185],
   [46.0665558, 11.1180829],
   [46.0665522, 11.118046],
   [46.0665479, 11.1180031],
   [46.066504, 11.1175601],
   [46.0664974, 11.117434]],
  'count_complessivo': 3521},
 22986312: {'coords': [[46.0674662, 11.1266008],
   [46.0675156, 11.1264331],
   [46.0675748, 11.1263186],
   [46.0678665, 11.1257547]],
  'count_complessivo': 2309},
 22986560: {'coords': [[46.069021, 11.1225351],
   [46.0693306, 11.1224874],
   [46.0695838, 11.1225012],
   [46.0698051, 11.1225938],
   [46.0698354, 11.1226063]],
  'count_complessivo': 1583},
 23056771: {'coords': [[46.0969958, 11.1125195],
   [46.096235, 11.1127734],
   [46.0955786, 11.1130632],
   [46.0944013, 11.1137155]],
  'count_complessivo': 26},
 23058066: {'coords': [[46.0838565, 11.1146839],
   [46.0839814, 11.1147368],
   [46.0840528, 11.114791],
   [46.0840863, 11.1148243]],
  'count_complessivo': 287},
 23058125: {'coords': [[46.0903875, 11.1183169],
   [46.09

In [36]:
with open('ways_id.json', 'w') as f:
    json.dump(d, f)

### Map selecting month

In [14]:
df = pd.read_parquet('../produced_datasets/map_matched_edges_ids.parquet')
df = df.reset_index()
f = open('../produced_datasets/ways_id.json')
d = json.load(f)

In [15]:
def compute_route_freq(df, month, year): 
    '''
    helper function to compute the routes and their frequency for the selected month
    '''
    data_selected_month = df.loc[(df.month == month) & (df.year == year)]
    data_selected_month = data_selected_month.reset_index()
    
    list_ways = []
    for row in range(len(data_selected_month)):
        list_ways.extend(data_selected_month.edge_ids[row])
    count_freq = Counter(list_ways) # count how many times passed through
    list_ways = list(set(list_ways))
    
    routes = []
    cnt = []
    for road in list_ways:
        route = d[road]['coords'] # coordinates (from way_id.json)
        routes.append(route)
        cnt.append(count_freq[road]) # num. times passed through (for the given month -> from counter above)
        
    return routes, cnt


def render_map(data=df, month=2, year=2021):
    '''
    render the map 
    '''
    routes, cnt = compute_route_freq(data, month, year) # helper function
    
    f_map = folium.Map(location=[46.066,11.133], tiles='cartodbdark_matter', zoom_start=14) #  tiles="cartodbdark_matter" "Stamen Toner" "OpenStreetMap"

    colormap = cm.LinearColormap(colors=['cyan', 'yellow', 'orange', 'red'], # ['darkblue', 'blue', 'cyan', 'yellow', 'orange', 'red']
                                index = np.linspace(min(cnt), max(cnt), num=4),
                                vmin = min(cnt), vmax = max(cnt), 
                                caption='Number of times the segment was passed through')
        
    fg = folium.FeatureGroup(name='overpass_tratte_forti')    


    for way in range(len(routes)): 
        color = colormap(cnt[way])
        w = min(cnt[way]/100 , 5)
        folium.vector_layers.PolyLine(routes[way], color=color, weight=w).add_to(fg)


    f_map.add_child(fg)
    f_map.add_child(colormap)

    return f_map


In [18]:
import panel as pn
import param

# compute routes and their frequency for the selected month
def compute_route_freq(month, year): 
    '''
    helper function to compute the routes and their frequency for the selected month
    '''
    data_selected_month = df.loc[(df.month == month) & (df.year == year)]
    data_selected_month = data_selected_month.reset_index()
    
    list_ways = []
    for row in range(len(data_selected_month)):
        list_ways.extend(data_selected_month.edge_ids[row])
    count_freq = Counter(list_ways) # count how many times passed through
    list_ways = list(set(list_ways))
    
    routes = []
    cnt = []
    for road in list_ways:
        route = d[str(road)]['coords'] # coordinates (from way_id.json)
        routes.append(route)
        cnt.append(count_freq[road]) # num. times passed through (for the given month -> from counter above)
        
    return routes, cnt


# base map
def get_map():
    return folium.Map(location=[46.066667,11.133333], tiles='Stamen Toner', zoom_start=14) 

map = get_map()
# pn.panel(map, height=400)


# plot data on map
def plot_data(map, df, month, year):
    '''
    render the map 
    '''
    routes, cnt = compute_route_freq(month, year)# helper function
    
    colormap = cm.LinearColormap(colors=['yellow', 'orange', 'red', 'darkred'], 
                                index = np.linspace(min(cnt), max(cnt), num=4),
                                vmin = min(cnt), vmax = max(cnt), 
                                caption='Number of times the segment was passed through')
    
    fg = folium.FeatureGroup(name='overpass_tratte_forti')    

    for way in range(len(routes)): 
        color = colormap(cnt[way])
        w = min(cnt[way]/10, 5)
        folium.vector_layers.PolyLine(routes[way], color=color, weight=w).add_to(fg)

    map.add_child(fg)
    map.add_child(colormap)

    return map

In [28]:
# Parameterized -> add slider
class PanelFoliumMap(param.Parameterized):
    month = param.Integer(2, bounds=(1,12))
    year = param.Integer(2021, bounds=(2020,2022))
        
    def __init__(self, **params):
        super().__init__(**params)
        self.map = get_map()
        self.folium_pane = pn.pane.plot.Folium(sizing_mode="stretch_both", min_height=500, min_width=900, margin=0)    
        self.view = pn.Column(pn.Row(self.param.month, self.param.year), 
                              self.folium_pane) 
        self._update_map()

    @param.depends("month", "year", watch=True)
    def _update_map(self):
        self.map = get_map()
        df_month = compute_route_freq(month=self.month, year=self.year)
        plot_data(self.map, df_month, self.month, year=self.year)
        self.folium_pane.object = self.map

        
app = PanelFoliumMap()
app.view

Column
    [0] Row
        [0] IntSlider(end=12, name='Month', start=1, value=2)
        [1] IntSlider(end=2022, name='Year', start=2020, value=2021)
    [1] Folium(Map, margin=0, min_height=500, min_width=900, sizing_mode='stretch_both')

In [53]:
# app.view.save('panel.html')
pn.template.FastListTemplate(site="Panel", title="Folium", main=["Most Travelled Routes", PanelFoliumMap().view]).servable();
# as administrator:
# netstat -ano | findstr < Port Number aka 5006 >
# taskkill /F /PID < Process Id >
# anaconda:
# panel serve interactive_tratte_forti.ipynb

### Map selecting time of the day

In [44]:
df = pd.read_parquet('../produced_datasets/map_matched_edges_ids.parquet')
df = df.reset_index()
df.head()

,index,unique_id,route,matched,dates,month,year,edge_ids
0,0,00000bf0-0000-1000-a6a0-5b7d6a617b60_2020-11-30,"[[46.069691, 11.121241], [46.069716, 11.121509...",True,2020-11-30,11,2020,"[748981520, 748981521, 48769827, 72847149, 177..."
1,1,00001d90-0000-1000-8b30-5b7d6a617b60_2020-11-30,"[[46.07016, 11.120398], [46.070181, 11.12065],...",True,2020-11-30,11,2020,"[748981521, 24771360, 48769827, 72847149, 1771..."
2,2,0000210d-0000-1000-8d7d-5b7d6a617b60_2020-11-30,"[[46.067214, 11.126787], [46.067211, 11.126772...",True,2020-11-30,11,2020,"[308335488, 295396001, 574541250, 109372547, 1..."
3,4,00000e00-0000-1000-9ab0-5b7d6a617b60_2020-11-30,"[[46.069695, 11.121283], [46.069716, 11.121509...",True,2020-11-30,11,2020,"[227756256, 227756257, 24771362, 216745509, 24..."
4,7,00002538-0000-1000-8f68-5b7d6a617b60_2020-11-30,"[[46.067217, 11.126806], [46.067221, 11.126829]]",True,2020-11-30,11,2020,[153939403]


In [45]:
# get timestamps 
orig_df= pd.read_parquet('../data/trips_pointv3_cleaned.parquet')

In [46]:
new_df = orig_df.groupby('unique_id', as_index=False).aggregate({"point_timestamp":lambda x: x.to_list()})

In [47]:
start_time = {}
for i in range(len(new_df)):
    start_time[new_df.at[i, 'unique_id']] = min(new_df.at[i, 'point_timestamp'])

In [48]:
# add timestamp to map-matched dataframe
tmstmp = []
for row in range(len(df)):
    tmstmp.append(start_time[df.at[row, 'unique_id']])

In [49]:
df['start_time'] = tmstmp
df = df.drop(['index'], axis=1)
df.head(3)

,unique_id,route,matched,dates,month,year,edge_ids,start_time
0,00000bf0-0000-1000-a6a0-5b7d6a617b60_2020-11-30,"[[46.069691, 11.121241], [46.069716, 11.121509...",True,2020-11-30,11,2020,"[748981520, 748981521, 48769827, 72847149, 177...",2020-11-30 14:32:20+00:00
1,00001d90-0000-1000-8b30-5b7d6a617b60_2020-11-30,"[[46.07016, 11.120398], [46.070181, 11.12065],...",True,2020-11-30,11,2020,"[748981521, 24771360, 48769827, 72847149, 1771...",2020-11-30 15:37:44+00:00
2,0000210d-0000-1000-8d7d-5b7d6a617b60_2020-11-30,"[[46.067214, 11.126787], [46.067211, 11.126772...",True,2020-11-30,11,2020,"[308335488, 295396001, 574541250, 109372547, 1...",2020-11-30 15:42:50+00:00


In [50]:
f = open('../produced_datasets/ways_id.json')
d = json.load(f) # way_id : { coords: [list of lists], count_complessivo: int }

In [51]:
def compute_route_freq(start_h, end_h): 
    '''
    helper function to compute the routes and their frequency for the timewindow
    '''
    data_selected = df.loc[(df.start_time.dt.hour >= start_h) & (df.start_time.dt.hour < end_h)]
    data_selected = data_selected.reset_index()
    
    list_ways = []
    for row in range(len(data_selected)):
        list_ways.extend(data_selected.edge_ids[row])
    count_freq = Counter(list_ways) # count how many times passed through
    list_ways = list(set(list_ways))
    
    routes = []
    cnt = []
    for road in list_ways:
        route = d[str(road)]['coords'] # coordinates (from way_id.json)
        routes.append(route)
        cnt.append(count_freq[road]) # num. times passed through (for the given month -> from counter above)
        
    return routes, cnt


def get_map():
    return folium.Map(location=[46.066667,11.133333], tiles='Stamen Toner', zoom_start=14) 

map = get_map()

def render_map(map, df, month, year):
    '''
    render the map 
    '''
    routes, cnt = compute_route_freq(month, year)# helper function
    
    colormap = cm.LinearColormap(colors=['yellow', 'orange', 'red', 'darkred'], 
                                index = np.linspace(min(cnt), max(cnt), num=4),
                                vmin = min(cnt), vmax = max(cnt), 
                                caption='Number of times the segment was passed through')
    
    fg = folium.FeatureGroup(name='overpass_tratte_forti')    

    for way in range(len(routes)): 
        color = colormap(cnt[way])
        w = min(cnt[way]/10, 5)
        folium.vector_layers.PolyLine(routes[way], color=color, weight=w).add_to(fg)

    map.add_child(fg)
    map.add_child(colormap)

    return map

In [ ]:
# Parameterized -> add slider
pn.extension()

class PanelFoliumMap(param.Parameterized):
    start_hour = param.Integer(14, bounds=(0,23))
    end_hour = param.Integer(15, bounds=(0,23))
        
    def __init__(self, **params):
        super().__init__(**params)
        self.map = get_map()
        self.folium_pane = pn.pane.plot.Folium(sizing_mode="stretch_both", min_height=500, min_width=900, margin=0)     
        self.view = pn.Column(pn.Row(self.param.start_hour, self.param.end_hour), 
                              self.folium_pane) 
        self._update_map()

    @param.depends("start_hour", "end_hour", watch=True)
    def _update_map(self):
        self.map = get_map()
        df_select = compute_route_freq(start_h=self.start_hour, end_h=self.end_hour)
        render_map(self.map, df_select, self.start_hour, self.end_hour)
        self.folium_pane.object = self.map

        
app = PanelFoliumMap()
app.view